# Lab 03 — Competitive learning and SOM

**Real dataset:** [Beothuk/uci-har-federated](https://huggingface.co/datasets/Beothuk/uci-har-federated). We use a small feature subset from human-activity recognition and ignore labels during SOM training. A validation split compares neighborhood settings; the untouched test split is used only for the final, fixed comparison.

Goal: build a simple winner-take-all prototype learner, extend it to a 2-D SOM, and quantify topology/cluster organization.

## The math behind the experiment

For an input $x$, competitive learning chooses the prototype with the smallest distance:

$$
k^*=\arg\min_k\|x-w_k\|_2.
$$

The winning prototype then moves toward the example:

$$
w_{k^*}\leftarrow w_{k^*}+\eta(x-w_{k^*}).
$$

A Self-Organizing Map extends this idea: nearby map units can also move, usually with a distance-dependent neighborhood weight $h(k,k^*)$.

The important idea is simple: **the model is not given class labels; it organizes examples by similarity.**

In [ ]:
!pip -q install datasets scikit-learn pandas numpy matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd
from sklearn.preprocessing import StandardScaler
raw=load_dataset("Beothuk/uci-har-federated",split="train[:1500]")
print(raw.features)
# Select numeric features; make deterministic train/validation/test partitions before fitting preprocessing.
rows=raw.to_pandas(); num=rows.select_dtypes(include=np.number)
Xraw=num.iloc[:,:min(12,num.shape[1])].fillna(0).values
split_rng=np.random.default_rng(17)
perm=split_rng.permutation(len(Xraw))
n_train=int(.70*len(Xraw)); n_val=int(.15*len(Xraw))
train_idx=perm[:n_train]; val_idx=perm[n_train:n_train+n_val]; test_idx=perm[n_train+n_val:]
scaler=StandardScaler().fit(Xraw[train_idx])
X=scaler.transform(Xraw)
X_train,X_val,X_test=X[train_idx],X[val_idx],X[test_idx]
print("train",X_train.shape,"validation",X_val.shape,"test",X_test.shape,"numeric columns",list(num.columns[:min(12,num.shape[1])]))

## Step 1 — Predict

A winner-take-all learner should place prototypes near dense regions. A SOM should additionally make neighboring map cells represent nearby input regions. It is **not** expected to discover labels directly because labels are not used during training.

In [ ]:
def train_prototypes(X,k=8,epochs=8,lr=.2,seed=0):
 rng=np.random.default_rng(seed); W=X[rng.choice(len(X),k,replace=False)].copy()
 for _ in range(epochs):
  for x in X[rng.permutation(len(X))]:
   j=np.argmin(((W-x)**2).sum(1)); W[j]+=lr*(x-W[j])
 return W
W=train_prototypes(X_train); print("prototype shape",W.shape)
# Quantization error: average distance to winning prototype.
qe=np.mean([np.min(((W-x)**2).sum(1))**.5 for x in X_val]); print("validation_quantization_error",qe)

In [ ]:
def train_som(X,shape=(4,4),epochs=10,lr=.25,sigma=2.,seed=0):
 rng=np.random.default_rng(seed); n=shape[0]*shape[1]; W=X[rng.choice(len(X),n,replace=False)].copy(); coords=np.array([(i,j) for i in range(shape[0]) for j in range(shape[1])])
 for e in range(epochs):
  a=lr*(1-e/epochs); s=max(.5,sigma*(1-e/epochs))
  for x in X[rng.permutation(len(X))]:
   d=((W-x)**2).sum(1); win=np.argmin(d); grid=((coords-coords[win])**2).sum(1); h=np.exp(-grid/(2*s*s)); W += a*h[:,None]*(x-W)
 return W
S=train_som(X_train)
print("SOM validation quantization error",np.mean([np.min(((S-x)**2).sum(1))**.5 for x in X_val]))

## Step 2 — Intervention

Change only the SOM neighborhood width. A very broad neighborhood should encourage smooth maps but can reduce local specialization; a narrow neighborhood can fit local structure while losing global ordering.

In [ ]:
rows=[]
for sig in [.5,1.,2.,4.]:
 S=train_som(X_train,sigma=sig,seed=0); qe=np.mean([np.min(((S-x)**2).sum(1))**.5 for x in X_val]); rows.append([sig,qe])
results=pd.DataFrame(rows,columns=["initial_sigma","validation_quantization_error"]); results

## Read the graph — map quality has trade-offs

This graph shows validation quantization error for different initial neighborhood widths. Lower error means each example is closer to some learned prototype, but **lower quantization error alone is not enough to prove a better-organized map**. Also inspect the map geometry and test whether nearby input examples map to nearby grid cells.

Because the neighborhood width changes the update rule, the curve need not be monotonic. Explain the observed trend from the experiment rather than assuming one setting must win.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(results["initial_sigma"], results["validation_quantization_error"], marker="o")
ax.set_title("SOM neighborhood width vs validation quantization error")
ax.set_xlabel("Initial neighborhood width (sigma)")
ax.set_ylabel("Held-out quantization error (lower is closer fit)")
ax.grid(alpha=.3)
for _, row in results.iterrows():
    ax.annotate(f'{row["validation_quantization_error"]:.3f}',
                (row["initial_sigma"], row["validation_quantization_error"]),
                xytext=(4, 6), textcoords="offset points", fontsize=8)
plt.tight_layout()
plt.show()

## Step 3 — A minimal growing 1-D SOM variant

A fixed SOM starts with a predetermined number of units. This educational growing-map variant can insert a unit when a training example is farther from its best-matching unit than a threshold. Otherwise, it updates the winner and nearby units. The threshold is set from the 80th percentile of initial training-set distances, and a maximum unit count prevents unbounded growth.

This is a small, explicit growing-SOM-family demonstration, not a claim to reproduce one canonical Evolving SOM algorithm. Real variants differ in growth, pruning, and graph-update rules.

In [ ]:
def train_growing_som(X, initial_units=4, max_units=24, epochs=8, lr=.2, sigma=1.5, seed=0):
    rng=np.random.default_rng(seed)
    W=X[rng.choice(len(X),initial_units,replace=False)].copy()
    initial_dist=np.min(np.linalg.norm(X[:,None,:]-W[None,:,:],axis=2),axis=1)
    threshold=float(np.quantile(initial_dist,.80))
    for epoch in range(epochs):
        rate=lr*(1-epoch/epochs)
        width=max(.5,sigma*(1-epoch/epochs))
        for x in X[rng.permutation(len(X))]:
            distances=np.linalg.norm(W-x,axis=1)
            winner=int(np.argmin(distances))
            if distances[winner] > threshold and len(W) < max_units:
                W=np.insert(W,winner+1,x,axis=0)
            else:
                positions=np.arange(len(W))
                neighborhood=np.exp(-((positions-winner)**2)/(2*width**2))
                W += rate*neighborhood[:,None]*(x-W)
    return W,threshold

def quantization_error(W, examples):
    distances=np.linalg.norm(examples[:,None,:]-W[None,:,:],axis=2)
    return float(distances.min(axis=1).mean())

fixed_W=train_som(X_train,shape=(4,4),epochs=10,seed=0)
growing_W,growth_threshold=train_growing_som(X_train,seed=0)
comparison=pd.DataFrame([
    ["Fixed 4x4 SOM",len(fixed_W),quantization_error(fixed_W,X_val)],
    ["Growing 1-D SOM variant",len(growing_W),quantization_error(growing_W,X_val)]
],columns=["model","units","validation_quantization_error"])
print("Growth threshold:",round(growth_threshold,4))
comparison

### Interpret the capacity trade-off

Compare held-out error and unit count. If the growing map improves error, ask whether the improvement justifies the extra units and whether it persists across seeds. If it does not improve, possible explanations include an unsuitable threshold, a weak growth rule, or data that do not benefit from adaptive capacity.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5,4.5))
ax.bar(comparison["model"],comparison["validation_quantization_error"])
ax.set_title("Fixed vs growing SOM: validation quantization error")
ax.set_ylabel("Held-out quantization error (lower is better)")
ax.tick_params(axis="x",rotation=10)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.5,4.5))
ax.bar(comparison["model"],comparison["units"])
ax.set_title("Capacity used by each map")
ax.set_ylabel("Number of units")
ax.tick_params(axis="x",rotation=10)
plt.tight_layout()
plt.show()

## Final check on the untouched test split

Choose the neighborhood width using validation quantization error only. Then refit the fixed SOM on the combined training-plus-validation examples using that width. The growing-map variant keeps its predeclared growth rule. Finally, measure both maps on the untouched test examples; do not change either model after seeing these numbers.


In [ ]:
best_sigma=float(results.loc[results["validation_quantization_error"].idxmin(),"initial_sigma"])
X_fit=np.vstack([X_train,X_val])
fixed_final=train_som(X_fit,shape=(4,4),epochs=10,sigma=best_sigma,seed=0)
growing_final,_=train_growing_som(X_fit,seed=0)
final_test=pd.DataFrame([
    ["Fixed 4x4 SOM",quantization_error(fixed_final,X_test)],
    ["Growing 1-D SOM variant",quantization_error(growing_final,X_test)]
],columns=["model","final_test_quantization_error"])
print("Validation-selected initial sigma:",best_sigma)
final_test


## Conclusion / answer key

Expected: prototype learning reduces quantization error compared with random prototypes; SOM neighborhood choice changes the trade-off between local fit and map smoothness. The growing-map variant may use more units to reduce validation quantization error, but this is not guaranteed. Compare error and capacity, and repeat with multiple seeds before making a general claim.

### Research extension
Compare SOM map sizes at a fixed prototype count and evaluate topology preservation with a neighborhood-preservation metric. For the growing variant, sweep the growth threshold and report held-out error, number of units, and runtime across at least three seeds.